<a href="https://colab.research.google.com/github/RafaelaMlucca/AnaliseViolMulher/blob/main/03_pre_processamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Pré-processamento

**Projeto:** Assinaturas preditivas dos tipos de violência contra a mulher  
**Autora:** Rafaela Lucca

Pré-processamento mínimo, baseado em decisões tomadas após a EDA.


## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/projeto_violencia_mulher')

# Carrega a base que veio do notebook 01
df = pd.read_parquet(DRIVE / 'viol_mulher.parquet')
print(f'Base carregada: {df.shape}')

Base carregada: (1677888, 57)


---

## 1. Reconstruir idade em anos

A coluna `NU_IDADE_N` codifica idade com prefixo da unidade (1=h, 2=d, 3=m, 4=ano).  
Ex.: `'4018'` = 18 anos. Vamos converter para anos completos.

In [3]:
def idade_em_anos(codigo):
    if pd.isna(codigo):
        return np.nan
    s = str(codigo).zfill(4)
    if s[0] == '4':
        return int(s[1:])
    elif s[0] in ('1', '2', '3'):  # hora, dia, mês -> menos de 1 ano completo
        return 0
    return np.nan

df['IDADE'] = df['NU_IDADE_N'].apply(idade_em_anos)
print(f'Idade reconstruída. Range: {df["IDADE"].min()} a {df["IDADE"].max()}')

Idade reconstruída. Range: 0.0 a 130.0


## 2. Truncar idades absurdas


In [4]:
n_absurdo = (df['IDADE'] > 110).sum()
df.loc[df['IDADE'] > 110, 'IDADE'] = np.nan
print(f'Idades > 110 truncadas para NaN: {n_absurdo}')
print(f'Novo range: {df["IDADE"].min()} a {df["IDADE"].max()}')

Idades > 110 truncadas para NaN: 39
Novo range: 0.0 a 110.0


## 4. Remover duplicatas


In [6]:
n_antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_depois = len(df)

n_dup = n_antes - n_depois
print(f'Duplicatas removidas: {n_dup:,} ({100*n_dup/n_antes:.3f}%)')
print(f'Base após dedup: {df.shape}')

Duplicatas removidas: 938 (0.056%)
Base após dedup: (1676950, 58)


## 5. Verificar inconsistências temporais

Data de ocorrência não pode ser posterior à data de notificação.

In [7]:
# Converte datas (estão como string YYYYMMDD)
dt_ocor = pd.to_datetime(df['DT_OCOR'], format='%Y%m%d', errors='coerce')
dt_notif = pd.to_datetime(df['DT_NOTIFIC'], format='%Y%m%d', errors='coerce')

# Inconsistência: ocorrência depois da notificação
mask_invalido = dt_ocor > dt_notif
n_invalido = mask_invalido.sum()
print(f'Casos com DT_OCOR > DT_NOTIFIC: {n_invalido:,} ({100*n_invalido/len(df):.2f}%)')

# Decisão: remover (pois é fisicamente impossível)
df = df[~mask_invalido].reset_index(drop=True)
print(f'Base após remoção: {df.shape}')

Casos com DT_OCOR > DT_NOTIFIC: 4 (0.00%)
Base após remoção: (1676946, 58)


## 6. Verificação rápida — `head()` da base limpa

In [9]:
# Resumo final
print(f'Shape final: {df.shape}')
print(f'\nDistribuição dos alvos:')
for col in ['y_fisic', 'y_psico', 'y_sexu']:
    n_pos = (df[col] == 1).sum()
    n_neg = (df[col] == 0).sum()
    n_na = df[col].isna().sum()
    prev = 100 * n_pos / (n_pos + n_neg)
    print(f'  {col}: Sim={n_pos:>7,} | Não={n_neg:>7,} | NaN={n_na:>6,} | Prev={prev:.1f}%')

Shape final: (1676946, 58)

Distribuição dos alvos:
  y_fisic: Sim=827,260 | Não=828,346 | NaN=21,340 | Prev=50.0%
  y_psico: Sim=384,131 | Não=1,252,927 | NaN=39,888 | Prev=23.5%
  y_sexu: Sim=271,174 | Não=1,365,041 | NaN=40,731 | Prev=16.6%


## 7. Salvar base limpa

In [ ]:
df.to_parquet(DRIVE / 'viol_mulher_limpa.parquet', index=False)
print(f'Salvo: {DRIVE / "viol_mulher_limpa.parquet"}')